# ALQAC 2026 — Private Inference
Notebook production: private input → cached retrieval → inference → validated submission. Không tự upload leaderboard.

In [ ]:
from pathlib import Path
import os

IS_KAGGLE = Path('/kaggle').exists()
IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ
PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / 'pyproject.toml').exists(), 'Run notebook from repository root'

In [ ]:
%pip install -q -e .

In [ ]:
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ['ALQAC_TEAM_TOKEN'] = UserSecretsClient().get_secret('ALQAC_TEAM_TOKEN')
elif IS_COLAB:
    from google.colab import userdata
    os.environ['ALQAC_TEAM_TOKEN'] = userdata.get('ALQAC_TEAM_TOKEN')
else:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env')
assert os.getenv('ALQAC_TEAM_TOKEN'), 'Missing ALQAC_TEAM_TOKEN'

In [ ]:
PRIVATE_INPUT = Path('/kaggle/input/alqac2026-private/private_test.json') if IS_KAGGLE else PROJECT_ROOT / 'data/private/private_test.json'
assert PRIVATE_INPUT.exists(), f'Missing private input: {PRIVATE_INPUT}'
RUN_MODE = 'smoke'  # smoke | full
assert RUN_MODE in {'smoke', 'full'}
LIMIT = 2 if RUN_MODE == 'smoke' else None
OUTPUT_DIR = Path('/kaggle/working/alqac2026/submissions') if IS_KAGGLE else PROJECT_ROOT / 'submissions'
RUN_DIR = OUTPUT_DIR / f'private_candidate_{RUN_MODE}'

In [ ]:
from alqac2026.runner import run_experiment

result = run_experiment(
    config_path=PROJECT_ROOT / 'configs/candidate.yaml',
    input_path=PRIVATE_INPUT,
    resume_run=RUN_DIR,
    limit=LIMIT,
)
assert result['validation']['status'] == 'PASS'
result